# Requirement 2 - Multiple Campaigns

This notebook is the final notebook for Requirement 2. It is aligned with the current repository code and uses the saved Req2 results and figures already present in `data/picklefiles/` and `outputs/req2/`.

Requirement 2 extends the bidding problem to multiple campaigns with a shared budget and a conflict graph. The learner selects a feasible joint action: it may bid on several compatible campaigns, but never on both endpoints of a conflict edge in the same round.


In [ ]:
from pathlib import Path
import pickle
import sys

import numpy as np
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "utils").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data" / "picklefiles"
OUTPUTS_DIR = ROOT / "outputs"

def load_pickle(name):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run the corresponding experiment first.")
    with path.open("rb") as f:
        return pickle.load(f)

def show_png(relative_path, width=900):
    path = OUTPUTS_DIR / relative_path
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f"**Missing plot:** `{path}`"))

def markdown_table(rows, columns, formats=None):
    formats = formats or {}
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in rows:
        vals = []
        for col in columns:
            value = row[col]
            fmt = formats.get(col)
            vals.append(fmt.format(value) if fmt else str(value))
        lines.append("| " + " | ".join(vals) + " |")
    display(Markdown("\n".join(lines)))


## Code Path and Parameters

The authoritative implementation for this requirement is `utils/run_req2.py`. The agent is `CombinatorialUCBAgent`, which combines UCB-style optimistic utility estimates with a joint LP over feasible campaign/bid actions.


In [ ]:
from utils.environments import MultiCampaignEnv
from utils.experiments import compute_clairvoyant_multi
from utils.run_req2 import (
    AVAILABLE_BIDS,
    BUDGET,
    CONFLICT_EDGES,
    N_COMPETITORS,
    N_TRIALS,
    T,
    VALUES,
)

env = MultiCampaignEnv(
    values=VALUES,
    budget=BUDGET,
    T=T,
    available_bids=AVAILABLE_BIDS,
    n_competitors=N_COMPETITORS,
    conflict_edges=CONFLICT_EDGES,
    seed=0,
)

parameter_rows = [
    {"quantity": "campaign values", "value": VALUES},
    {"quantity": "bid-set sizes Ks", "value": env.Ks},
    {"quantity": "horizon T", "value": T},
    {"quantity": "trials", "value": N_TRIALS},
    {"quantity": "budget B", "value": BUDGET},
    {"quantity": "rho = B/T", "value": BUDGET / T},
    {"quantity": "conflict edges", "value": CONFLICT_EDGES},
]
markdown_table(parameter_rows, ["quantity", "value"])


## Clairvoyant LP

The clairvoyant benchmark knows the true win probabilities of each campaign/bid pair. It solves a joint LP over feasible actions, respecting both the shared budget rate and the conflict graph.


In [ ]:
win_prob_list = env.win_probabilities()
clairvoyant_marginals, opt_utility = compute_clairvoyant_multi(
    values=np.array(VALUES),
    bid_sets=env.bid_sets,
    rho=BUDGET / T,
    win_prob_list=win_prob_list,
    conflict_edges=CONFLICT_EDGES,
)

rows = [{
    "metric": "clairvoyant utility/round",
    "value": opt_utility,
}, {
    "metric": "rho",
    "value": BUDGET / T,
}, {
    "metric": "number of campaigns",
    "value": env.N,
}]
markdown_table(rows, ["metric", "value"], {"value": "{:.5f}"})


## Campaign Bid Table

The table below shows the expected utility/cost of each admissible campaign bid, plus the clairvoyant marginal probability induced by the joint LP solution.


In [ ]:
bid_rows = []
for i in range(env.N):
    for bid, win_p, marginal in zip(env.bid_sets[i], win_prob_list[i], clairvoyant_marginals[i]):
        bid_rows.append({
            "campaign": i,
            "value": VALUES[i],
            "bid": bid,
            "P(win)": win_p,
            "exp utility": (VALUES[i] - bid) * win_p,
            "exp cost": bid * win_p,
            "clairvoyant marginal": marginal,
        })
markdown_table(
    bid_rows,
    ["campaign", "value", "bid", "P(win)", "exp utility", "exp cost", "clairvoyant marginal"],
    {"value": "{:.2f}", "bid": "{:.2f}", "P(win)": "{:.4f}", "exp utility": "{:.4f}", "exp cost": "{:.4f}", "clairvoyant marginal": "{:.4f}"},
)


## Saved Results Summary

The saved Req2 result is the output of the 20-trial combinatorial UCB experiment. It reports cumulative pseudo-regret and cumulative cost against the clairvoyant LP benchmark.


In [ ]:
result = load_pickle("req2_comb_ucb_results.pkl")
final_regret = float(result["mean_regret"][-1])
final_cost = float(result["mean_cumcost"][-1])
summary_rows = [{
    "trials": result["n_trials"],
    "T": len(result["mean_regret"]),
    "final regret": final_regret,
    "avg regret": final_regret / len(result["mean_regret"]),
    "final cost": final_cost,
    "budget slack": BUDGET - final_cost,
    "mean cost/round": final_cost / len(result["mean_cumcost"]),
}]
markdown_table(
    summary_rows,
    ["trials", "T", "final regret", "avg regret", "final cost", "budget slack", "mean cost/round"],
    {"final regret": "{:.3f}", "avg regret": "{:.5f}", "final cost": "{:.3f}", "budget slack": "{:.3f}", "mean cost/round": "{:.5f}"},
)


## No-Regret and Budget Checks

The expected theoretical pattern is sublinear pseudo-regret: cumulative regret grows, but average regret should decrease. At the same time, cumulative cost should stay close to the shared budget without systematically exceeding it.


In [ ]:
ts = [100, 500, 1000, 2500, 5000, 7500, 10000]
rows = []
for t in ts:
    rows.append({
        "t": t,
        "mean regret": float(result["mean_regret"][t - 1]),
        "avg regret": float(result["mean_regret"][t - 1] / t),
        "cum cost": float(result["mean_cumcost"][t - 1]),
        "target budget": float((BUDGET / T) * t),
    })
markdown_table(
    rows,
    ["t", "mean regret", "avg regret", "cum cost", "target budget"],
    {"mean regret": "{:.3f}", "avg regret": "{:.5f}", "cum cost": "{:.3f}", "target budget": "{:.3f}"},
)


## Req2 Figures

These plots are generated by the Req2 pipeline and stored in `outputs/req2/`.


In [ ]:
show_png("req2/regret.png", width=900)
show_png("req2/regret_annotated.png", width=900)
show_png("req2/average_regret.png", width=900)
show_png("req2/budget.png", width=900)
show_png("req2/highest_competing_bid_distributions.png", width=950)
show_png("req2/pairwise_joint_bid_distributions.png", width=950)


## Interpretation

The Req2 result is coherent when the final cumulative spend is near `B = 1600`, the average regret decreases over time, and the stochastic diagnostics match independent highest-bid distributions. This is exactly the qualitative behavior expected from a budget-aware combinatorial UCB learner in a stochastic setting.


## Additional Explainability Figures

These figures are stored under `outputs/explainability/req2/`. They are designed for interpretation: budget pacing, per-campaign cost, per-campaign wins, bid frequencies, win rates by bid, active campaign patterns, and per-campaign outcome scatter plots.


In [ ]:
for rel in [
    "explainability/req2/01_mean_cumulative_cost_vs_pacing.png",
    "explainability/req2/02_campaign_cumulative_cost_seed7.png",
    "explainability/req2/03_campaign_cumulative_wins_seed7.png",
    "explainability/req2/04_campaign_bid_frequencies_seed7.png",
    "explainability/req2/05_campaign_win_rate_by_bid_seed7.png",
    "explainability/req2/06_activation_heatmap_first500_seed7.png",
    "explainability/req2/07_campaign_outcome_scatter_seed7.png",
    "explainability/req2/08_campaign_rolling_metrics_seed7.png",
]:
    display(Markdown(f"### `{rel}`"))
    show_png(rel, width=950)


## Optional: Regenerate Req2 Outputs

Run this only if you intentionally want to recompute the full experiment. It can take several minutes because the agent solves a joint LP at every round.


In [ ]:
# from utils.run_req2 import run_req2
# run_req2()
